# BER: Business Entity Resolution on Kaggle (GPU)

Settings (right panel): **Accelerator = GPU T4 x2**, **Internet = On** (to clone the code), and
**Persistence = Files** if you want `models/` to survive a restart. Add the competition data as a
Kaggle dataset (the `student_resource` folder). The paths are found automatically below.

Budget: training takes ~2.5-3.5 h (blocking + features ~1.5 h, GBMs on GPU, cross-encoder ~30-45 min),
and inference ~4-4.5 h. Run training and inference in one session, or save `models/` in between.

In [ ]:
!nvidia-smi -L
!free -g | head -2; nproc; df -h /kaggle/working | tail -1

## 1. Code

In [ ]:
import os
REPO = "https://github.com/MounishSenisetty/BER.git"   # private repo: https://<user>:<token>@github.com/...
BRANCH = "main"
CODE = "/kaggle/working/BER"
if not os.path.exists(CODE):
    # GIT_LFS_SKIP_SMUDGE: skip the 2.4 GB dataset stored in the repo (the Kaggle dataset is used if attached)
    !GIT_LFS_SKIP_SMUDGE=1 git clone -q --depth 1 -b $BRANCH $REPO $CODE
!cd $CODE && git log --oneline -1

In [ ]:
# torch, transformers, xgboost (CUDA), lightgbm, scikit-learn, pandas are preinstalled on the Kaggle GPU image
!pip install -q rapidfuzz jellyfish faiss-cpu sentencepiece 2>&1 | tail -1
import torch, xgboost
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.device_count(), "GPU(s)")
print("xgboost", xgboost.__version__, "CUDA build:", xgboost.build_info().get("USE_CUDA"))

## 2. Data paths

In [ ]:
import glob, subprocess
hits = glob.glob("/kaggle/input/**/train/train_source1.tsv", recursive=True)
if hits:                                                  # competition data attached as a Kaggle dataset
    DATA = os.path.dirname(os.path.dirname(hits[0]))      # .../student_resource/dataset
else:                                                     # fall back to the copy stored in the repo (Git LFS)
    subprocess.run(f"cd {CODE} && git lfs install --local && git lfs pull", shell=True, check=True)
    DATA = f"{CODE}/student_resource/dataset"
TRAIN, TEST = f"{DATA}/train", f"{DATA}/test"
ROOT = os.path.dirname(DATA)                              # .../student_resource (has utils/)
WORK = "/kaggle/working"
os.environ.update(TRAIN=TRAIN, TEST=TEST, ROOT=ROOT, WORK=WORK, CODE=CODE)
print(TRAIN, TEST, sep="\n"); print(os.listdir(TRAIN), os.listdir(TEST))

## 3. Train

* S1 dropout is sized automatically from `../test` (test has more distractor records per entity).
* The cross-encoder turns on automatically with a CUDA GPU. Training keeps the best of
  stage1 / stage2 / stage2+ce by nested-CV macro F0.5.
* `CE_BACKBONE` picks the cross-encoder:
  * `"intfloat/e5-small-v2"` (MIT, 33M params) is a pretrained transformer, fine-tuned here. It's fast, and the default. It needs Internet to download the weights, or you can attach them as a dataset and give the local path.
  * `"microsoft/deberta-v3-small"` (MIT) is usually stronger but about 2-3x slower (roughly +1.5 h in total).
  * `""` is the small character transformer trained from scratch (no download).
* The log ends with the **loss report** (points lost per error type). Please share it.


In [ ]:
%%time
CE_BACKBONE = "intfloat/e5-small-v2"      # "" = character model trained from scratch
CE_ARGS = f"--ce-backbone {CE_BACKBONE}" if CE_BACKBONE else ""
# skips training when a model bundle already exists (e.g. after a restart with Persistence = Files)
if not os.path.exists(f"{WORK}/models/model_bundle.pkl"):
    !cd $CODE && python -u -m src.train --data-dir "$TRAIN" --model-dir $WORK/models --cache-dir /tmp/ber_s2 --chunk-records 200000 $CE_ARGS 2>&1 | tee $WORK/train.log | grep -vE "prepare \+ block \.\.\.$|features for .* \.\.\.$|  CE\[.\] step"
else:
    print("models/model_bundle.pkl exists - skipping training")

In [ ]:
import json
r = json.load(open(f"{WORK}/models/cv_report.json"))
print("blocking:", {k: v for k, v in r["blocking"].items() if k != "recall_at_cap"})
print("variants (nested-CV macro F0.5):", r.get("variants"), "-> final:", r.get("final"))
print("S1 dropout:", r.get("s1_drop"), "| lexicon entries:", r.get("lexicon_size"))
!sed -n '/Loss report/,/Examples FP/p' $WORK/train.log | head -40

## 4. Inference on the test split

In [ ]:
%%time
!cd $CODE && python -u -m src.inference --data-dir "$TEST" --model-dir $WORK/models --out-dir $WORK/output \
    --chunk-records 200000 2>&1 | tee $WORK/inference.log | grep -vE "prepare \+ block \.\.\.$|scoring for .* \.\.\.$"

## 5. Validate and package

In [ ]:
!python "$ROOT/utils/validate_submission.py" --matching $WORK/output/matching_results.tsv \
      --candidate $WORK/output/candidate_pairs.tsv --test-dir "$TEST"

In [ ]:
!cd $CODE && python scripts/make_submission_zip.py --team YOUR_TEAM --out-dir $WORK/output \
      --doc Documentation_template.md --dest $WORK
!ls -la $WORK/*.zip